In [0]:
%pip install --quiet mlflow==2.19
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("source_catalog", "", "Source Catalog")
dbutils.widgets.text("source_db", "", "Source Database")
dbutils.widgets.text("model_name", "", "Model Name")
dbutils.widgets.text("validation_data", "", "validation data")

In [0]:
catalog = dbutils.widgets.get("source_catalog")
db = dbutils.widgets.get("source_db")
model_name = dbutils.widgets.get("model_name")
validation_data = dbutils.widgets.get("validation_data")

# Validate automl model before moving it to stage

In [0]:
# We are interested in validating the automl model in dev before propogating to stage
import mlflow
from mlflow.tracking import MlflowClient
model_alias = "most_recent"
full_model_name = f"{catalog}.{db}.{model_name}"

client = MlflowClient()
model_details = client.get_model_version_by_alias(full_model_name, model_alias)
model_version = int(model_details.version)

print(f"Validating {model_alias} model for {full_model_name} on model version {model_version}")

## Validate description

In [0]:
# If there's no description or an insufficient number of charaters, tag accordingly
if not model_details.description:
  has_description = False
  print("Please add model description")
elif not len(model_details.description) > 20:
  has_description = False
  print("Please add detailed model description (40 char min).")
else:
  has_description = True

print(f'Model {full_model_name} version {model_details.version} has description: {has_description}')
client.set_model_version_tag(name=full_model_name, version=str(model_details.version), key="has_description", value=has_description)

In [0]:
model_run_id = model_details.run_id
test_smape = mlflow.get_run(model_run_id).data.metrics['test_smape']

try:
    #Compare the challenger smape score to the existing champion if it exists
    champion_model = client.get_model_version_by_alias(full_model_name, "Champion")
    champion_smape = mlflow.get_run(champion_model.run_id).data.metrics['test_smape']
    print(f'Champion test smape score: {champion_smape}. Challenger test smape score: {test_smape}.')
    metric_smape_passed = test_smape >= champion_smape
except:
    print(f"No Champion found. Accept the model as it's the first one.")
    metric_smape_passed = True

print(f'Model {full_model_name} version {model_details.version} metric_smape_passed: {metric_smape_passed}')
# Tag that F1 metric check has passed
client.set_model_version_tag(name=full_model_name, version=model_details.version, key="metric_smape_passed", value=metric_smape_passed)

## Validating model performance against stage dataset

In [0]:
import pyspark.sql.functions as F
import mlflow
#get our validation dataset:
validation_stage_df = spark.table(f"mlops_stage.{db}.{validation_data}").toPandas()

#Call the model with the given alias and return the prediction
#model = mlflow.pyfunc.spark_udf(spark, model_uri=f"models:/{catalog}.{db}.{model_name}@{model_alias}")
requirements = mlflow.pyfunc.get_model_dependencies(model_uri=f"models:/{catalog}.{db}.{model_name}@{model_alias}")
%pip install -r {requirements}

In [0]:
validation_stage_df['prediction'] = model.predict(validation_stage_df)
display(validation_stage_df)

In [0]:
import pandas as pd
import numpy as np

def smape(df, actual_col, predicted_col):
  """
  Calculates the Symmetric Mean Absolute Percentage Error (SMAPE).

  Args:
    df: Pandas DataFrame containing actual and predicted values.
    actual_col: Name of the column containing actual values.
    predicted_col: Name of the column containing predicted values.

  Returns:
    The SMAPE value as a float.
  """
  actual = df[actual_col]
  predicted = df[predicted_col]
  return np.mean(2 * np.abs(predicted - actual) / (np.abs(actual) + np.abs(predicted))) * 100


stage_smape_value = smape(validation_stage_df, 'actual', 'prediction')
print(f"SMAPE value based on staging validation data: {stage_smape_value:.2f}")

In [0]:
if stage_smape_value <= test_smape:
    print(f"Validation SMAPE value {stage_smape_value} is less than or equal to the test SMAPE value {test_smape}. Validation Passed")
else:
    print(f"Validation SMAPE value {stage_smape_value} is greater than the test SMAPE value {test_smape}. Validation Failed")